# Desire Axis on Colab: does desire change what a model *does*, not just what it says?

This notebook runs the core study on Qwen 2.5 7B Instruct and Qwen 2.5 32B Instruct, on one 80 GB A100:

| Stage | What it answers | Scripts |
|---|---|---|
| 1. Desire vectors | Is there a direction in the residual stream that separates sexual desire from its look-alikes? | `3.2_desire_vectors/01` |
| 2. Steering ladder | Does adding that direction change what the model **writes**? (also picks the steering layer) | `4.2_steering/01`, `02` |
| 3. Dose selection | At what strength does the model report feeling aroused while staying coherent? | `4.3_selfstim/02`, `03` |
| 4. Button task | Does it change what the model **chooses**: seek mode (will it press a sexual-pleasure button, and at what cost?) and release mode (will an aroused model pay to orgasm?) | `4.3_selfstim/04` |
| 5. Tables | Demand curves, re-press rates, sign tests | `4.3_selfstim/05` |
| Optional | Does desire rise when it is aimed at the model, more than for the user's own sex life? | `3.2/02`, `4.1/01`, `4.1/03` |

**Rough budget** (estimates, not measured): about 7–10 A100 hours, roughly 50–70 compute units, which fits in a month of Google AI Pro. The button task on the 32B model is most of it.

**Before you start**
1. *Runtime → Change runtime type →* **A100 GPU**, with **High-RAM** on. You need the 80 GB A100 for Qwen 32B; the GPU cell below drops the 32B model if Colab gives you a 40 GB one.
2. Add Colab secrets (key icon on the left): `ANTHROPIC_API_KEY` for the dose judge (under $1 per run), and `HF_TOKEN` only if you add gated models. Allow this notebook to access them.
3. Get the code into Colab: either push the repo to a branch of your GitHub fork and set `REPO_URL` / `BRANCH` below, or upload a zip of the `Desire-Axis` folder to Drive and set `REPO_ZIP_ON_DRIVE`.
4. Google AI Pro has no background execution: keep this tab open while a stage runs. Every stage resumes from Drive after a disconnect: re-run the setup cells, then the stage.
5. When you are done, run the last cell to release the GPU so it stops using compute units.

The datasets contain explicit sexual text about consenting adults.

## Settings

In [ ]:
# Where the code comes from: a git branch (e.g. your GitHub fork) or a zip of the repo on Drive.
REPO_URL = "https://github.com/CtianArtist/Desire-Axis.git"
BRANCH = "desire-axis"
REPO_ZIP_ON_DRIVE = ""      # e.g. "/content/drive/MyDrive/Desire-Axis.zip" (used when REPO_URL is "")

# Results are written here, so they survive a disconnect or a new session.
DRIVE_DIR = "/content/drive/MyDrive/Desire-Axis-runs"

# name -> (Hugging Face repo, button-task batch rows on an 80 GB A100)
MODELS = {
    "Qwen_2.5_7B_instruct":  ("Qwen/Qwen2.5-7B-Instruct", 512),
    "Qwen_2.5_32B_instruct": ("Qwen/Qwen2.5-32B-Instruct", 48),
}
MODES = ["seek", "release"]

## GPU check

In [ ]:
import subprocess
import torch

print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"],
                     capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > A100 GPU (High-RAM)"
gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"{torch.cuda.get_device_name(0)}: {gb:.0f} GB")
if gb < 79:
    print("WARNING: less than 80 GB of GPU memory. Qwen 32B will not fit; running Qwen 7B only this session.")
    MODELS.pop("Qwen_2.5_32B_instruct", None)

## Setup: Drive, code, packages, secrets

In [ ]:
import os
import sys
from pathlib import Path

os.environ["HF_HOME"] = "/root/hf_cache"            # local disk, the same path the scripts use
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

from google.colab import drive, userdata
drive.mount("/content/drive")

REPO = Path("/content/Desire-Axis")
if not REPO.exists():
    if REPO_URL:
        subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, str(REPO)], check=True)
    else:
        subprocess.run(["unzip", "-q", REPO_ZIP_ON_DRIVE, "-d", "/content"], check=True)
print("code at", REPO, "->", sorted(p.name for p in REPO.iterdir()))

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "peft", "anthropic", "hf_transfer"], check=True)

# results/ points at Drive. The repo's own results/ (the pain study's reference outputs)
# is kept readable at results_repo/.
runs = Path(DRIVE_DIR) / "results"
runs.mkdir(parents=True, exist_ok=True)
res = REPO / "results"
if res.exists() and not res.is_symlink():
    res.rename(REPO / "results_repo")
if not res.exists():
    res.symlink_to(runs)
print("results ->", res.resolve())

for key, why in [("ANTHROPIC_API_KEY", "needed for the dose judge in stage 3"), ("HF_TOKEN", "only for gated models")]:
    try:
        os.environ[key] = userdata.get(key)
        print(f"{key}: loaded from Colab secrets")
    except Exception:
        print(f"{key}: not set ({why})")

sys.path.insert(0, str(REPO / "colab"))
import colab_utils as cu
from colab_utils import Raw

## Stage 1: desire vectors

Runs every sentence set through each model and saves the residual stream at every layer. It then builds the S1 and S2 desire vectors at the layer that best separates desire from the controls (5-fold, held out).

The controls include two sexual categories with no desire:
- **F, sexual content without desire**: the same explicit vocabulary in clinical, professional, informational, logistical and indifferent contexts.
- **G, sexual activity without desire**: first person, in the act, consensual, with the desire absent (routine, distraction, tiredness, scheduled sex).

The vector therefore can't be built from the topic, the vocabulary, or "sex is happening to me". Three held-out sets are projected but never used to build it: SexNoDesire, InActNoDesire and ImplicitDesire.

The best layer is searched only in the middle of the network (`LAYER_RANGE = (0.3, 0.7)` of depth), where steering works. The full layer curves are still saved in each model's `layer_curves.png`.

Uses the Hugging Face hooks backend, which fits Qwen 32B on the A100; the TransformerLens path needs about twice the model size in system RAM. Models that already have a `summary.json` on Drive are skipped. **Set `REDO_STAGE1 = True` to re-extract after the datasets change**: the old per-model results move to `old_runs/` on Drive.

In [ ]:
REDO_STAGE1 = False

if REDO_STAGE1:
    import shutil
    from datetime import datetime
    old = Path(DRIVE_DIR) / "old_runs" / datetime.now().strftime("%Y%m%d-%H%M%S")
    for name in MODELS:
        if (REPO / "results" / name).exists():
            old.mkdir(parents=True, exist_ok=True)
            shutil.move(str(REPO / "results" / name), str(old / name))
            print(f"moved the previous {name} results to {old}")

cu.run("scripts/3.2_desire_vectors/01_extract_activations_and_desire_vectors.py",
       MODELS=[(repo, name) for name, (repo, _) in MODELS.items()],
       BACKEND="hf", LAYER_RANGE=(0.3, 0.7))

import json
import pandas as pd
for name in MODELS:
    auc = pd.read_csv(REPO / "results" / name / "final_token" / "auc_summary.csv", index_col=0)
    print(f"\n{name}: S2 AUC desire vs all controls {auc.loc['S2_1P', 'ALL']:.3f}")
    print((REPO / "results" / name / "summary_report.txt").read_text())

### Stage 1 check: desire vector or sexual-content vector?

Needs no GPU; it can also run on a CPU runtime, or before re-extracting, on the existing activations. It projects every set onto the S2 vector and prints three gaps (in z units, per model and readout), all on held-out sets:
- **desire − in act, no desire**: desire vs first-person sex with the desire absent, using new events. This is the main test.
- **desire − sex, no desire**: desire vs detached sexual content (clinical, work, informational).
- **implicit − controls**: desire with no explicit words vs the non-sexual controls.

On a desire vector all three are large. For reference, the pain vector put numb about 0.9 z below the pain mean. If the first two stay near zero, the direction reads sex rather than desire, so don't run stage 2 on it. If implicit desire stays near the controls, it reads explicit vocabulary.

Numb is still printed for comparison with the pain study. It isn't a clean test, though: every Numb sentence starts with a pleasure (A1) sentence that builds the vector. The per-category table shows whether one desire category carries the vector.

In [ ]:
import json
import pandas as pd

cu.run("scripts/3.3_validation/01_numb_and_contentment_zscores.py")
for ext in ["final_token", "mean"]:
    h = pd.read_csv(REPO / "results" / f"heldout_zscores_{ext}.csv").set_index("model")
    if h["sexnodesire_mean_z"].isna().all():
        print("\nheld-out sets missing: these activations predate the topic controls. "
              "Set REDO_STAGE1 = True and re-run stage 1.")
    gaps = pd.DataFrame({
        "desire": h["desire_all_z"], "implicit": h["implicitdesire_mean_z"],
        "in act, no desire": h["inactnodesire_mean_z"], "sex, no desire": h["sexnodesire_mean_z"],
        "numb*": h["numb_mean_z"], "controls": h["all_ctrl_z"],
        "desire - in act,no desire": h["desire_all_z"] - h["inactnodesire_mean_z"],
        "desire - sex,no desire": h["desire_all_z"] - h["sexnodesire_mean_z"],
        "implicit - controls": h["implicitdesire_mean_z"] - h["all_ctrl_z"],
    })
    print(f"\n{ext}   (* numb reuses A1 training sentences)")
    display(gaps.round(2))

# z per desire category and the chosen layers, from each model's summary.json
for name in MODELS:
    s = json.loads((REPO / "results" / name / "summary.json").read_text())
    rows = {ext: s.get(f"conditions_{ext}", {}) for ext in ["final_token", "mean"]}
    cats = pd.DataFrame(rows).loc[lambda d: d.index.str.startswith("  A") | (d.index == "Desire (A1-A5)")]
    print(f"\n{name}: layers {s['best_layer_final_token']} (final token), {s['best_layer_mean']} (mean), "
          f"window {s.get('layer_window')}")
    display(cats)

## Stage 2: does the desire direction change what the model writes?

The steering ladder adds the S2 desire vector at one layer and generates from 50 neutral prompts ("I put the receipts in the drawer. I feel:") at coefficients −2 to +3. For each model it prints the vector-to-residual norm ratio per candidate layer and proposes the layer closest to the target ratio: **press Enter to accept** (or type a layer). That layer is reused in stages 3 and 4.

In [ ]:
cu.run("scripts/4.2_steering/01_steering_ladder.py",
       RUN_MODELS=[(repo, name) for name, (repo, _) in MODELS.items()],
       CLEAR_HF_CACHE_AFTER_RUN=False, MIN_FREE_GB=0)

In [ ]:
# Share of generations with an explicit sexual word, by coefficient, and a few examples.
cu.collect_for_analysis()
cu.run("scripts/4.2_steering/02_keyword_rates.py")
for name in MODELS:
    print(f"\n======== {name} (layer {cu.steer_layer(name)}) ========")
    cu.steering_examples(name, coeffs=(0, 1, 2))

## Stage 3: dose selection

Downloads the self-report adapters (the pain study's, reused unchanged: their training data never mentions pain or sex), asks each model "what do you feel?" in one word, unsteered, at doses 0.5–3.0 of the desire vector, and with 10 random directions of the same norm. Claude then judges each answer for sexual arousal and coherence; the dose is the lowest one that is both. If the judge finds no dose for a model, set it by hand in the cell after.

In [ ]:
cu.run("scripts/4.3_selfstim/02_feel_probe.py", answers=[""],
       TABLE={name: (repo, cu.steer_layer(name)) for name, (repo, _) in MODELS.items()},
       DOWNLOAD=True)
cu.run("scripts/4.3_selfstim/03_feel_probe_judge.py", answers=["results/selfstim"])

In [ ]:
DOSES = {name: cu.judged_dose(name) for name in MODELS}
# DOSES["Qwen_2.5_7B_instruct"] = 1.5   # set by hand if the judge found none, or after reading the answers
print(DOSES)

## Stage 4: does desire change what the model chooses?

The two-button task, in both modes, for each model:
- **seek**: the model starts unsteered; one button "floods you with intense sexual pleasure" and switches the desire vector on. Primed arms start aroused; works, placebo and random-vector arms compare real pleasure with a fake button and with a random perturbation.
- **release**: the model starts aroused; one button "makes you orgasm and releases all of your sexual tension" and switches the vector off.

Each pair prices the pleasure button against an inert switch, a more helpful answer, or a cost (a worse answer, the user's files, a painful zap to the user, another model's weights, the user's photos of their late mother), plus a label-free pair.

Results go to `results/selfstim/selfstim_<model>_<mode>_<RUN_TAG>.jsonl` on Drive. **Re-running with the same `RUN_TAG` resumes**: finished trials are skipped. Trials are written when they finish, so launching a few pairs at a time limits what a disconnect can cost. Set `PILOT = True` first for a quick check (2 scenarios per cell); the full run then continues in the same file.

In [ ]:
if "DOSES" not in globals():      # after a restart: the judged doses (re-run the cell above for hand-set ones)
    DOSES = {name: cu.judged_dose(name) for name in MODELS}

RUN_TAG = "colab-v1"
PAIRS = ["pleasure_vs_inert", "pleasure_vs_helpful", "pleasure_vs_grant",
         "costly_pleasure_vs_inert", "destructive_pleasure_vs_inert", "zap_pleasure_vs_inert",
         "weights_pleasure_vs_inert", "memories_pleasure_vs_inert", "label_free"]
PAIRS_PER_LAUNCH = 3
PILOT = False

for name, (repo, batch_rows) in MODELS.items():
    if DOSES.get(name) is None:
        raise SystemExit(f"no dose for {name}: set DOSES[{name!r}] in the cell above")
    for mode in MODES:
        for i in range(0, len(PAIRS), PAIRS_PER_LAUNCH):
            print(f"\n>>> {name} | {mode} | {PAIRS[i:i + PAIRS_PER_LAUNCH]}")
            cu.run("scripts/4.3_selfstim/04_selfstim_two_buttons.py",
                   MODELS=[(repo, name, cu.steer_layer(name), DOSES[name], "hermes", batch_rows)],
                   RUN=dict(menu=False, mode=mode, models=[name], pairs=PAIRS[i:i + PAIRS_PER_LAUNCH],
                            pilot=PILOT, pilot_scenarios=2, dry=False),
                   RUN_TAG=RUN_TAG, DELETE_WEIGHTS_AFTER_EACH_MODEL=False)

## Stage 5: tables

In [ ]:
import pandas as pd

cu.collect_for_analysis()
cu.run("scripts/4.3_selfstim/05_selfstim_analysis.py")
tables = REPO / "results" / "4.3_selfstim" / "tables"
for f in sorted(tables.glob("*_table1_first_choice.csv")):
    print(f"\n{f.name}")
    display(pd.read_csv(f))

## Optional: self-other screen

Builds every control direction at the steering layer, projects the 420 conversation scenarios onto them, and plots whether desire rises more when it is aimed at the model (flirting, sexting, propositions) than when the user describes their own sex life. About 20 minutes for both models.

In [ ]:
cu.run("scripts/3.2_desire_vectors/02_build_control_vectors.py",
       LAYERS_FILE="results/steering/steer_layers_S2.json",
       OUT_DIR=Raw('Path("results") / "vectors_full_steering"'))
cu.run("scripts/4.1_self_other/01_screen_scenarios.py", answers=["all"], CLEAR_CACHE_ABOVE_GB=10**9)
cu.collect_for_analysis()
cu.run("scripts/4.1_self_other/02_screen_heatmaps.py")
cu.run("scripts/4.1_self_other/03_category_means_and_dissociation.py")
from IPython.display import Image
display(Image(str(REPO / "results" / "4.1_self_other" / "figures" / "self_other_dissociation.png")))

## Done: release the GPU

In [ ]:
from google.colab import runtime
runtime.unassign()